In [9]:
import tkinter as tk
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import mstats
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
import category_encoders as ce
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
   
def object_types_to_categorical(df):
    for j in df.select_dtypes(include='object'):
        df[j] = df[j].astype('category')
    return df

def descripe_data(df):
    return df.describe(include='all').loc['mean']

def show_nulls(df):
    for i in df:
        if (df[i].isnull().sum()/df.shape[0])*100 >50:
            print(df[i])

def show_categorical(df):
    cols=df.select_dtypes(include='category').columns.tolist()
    return cols

def simple_imputer(df,strategy):
    cols_with_null= df.columns[df.isnull().sum()>0]
    impute=SimpleImputer(strategy=strategy)
    impted=df[cols_with_null].select_dtypes(include='number')
    df[impted.columns]=impute.fit_transform(impted)
    return df

def fill_categoriacl(df,strategy,fill_value='None'):
    have_null= df.columns[df.isnull().sum()>0]
    categorical= df[have_null].select_dtypes(include='category')
    if strategy=='constant':
        imputer = SimpleImputer(strategy='constant' ,fill_value=fill_value)
    else:
        imputer = SimpleImputer(strategy='most_frequent')
    df[categorical.columns] = imputer.fit_transform(categorical)
    return df

def k_mean(df,n_value=5):
    cols_with_null= df.columns[df.isnull().sum()>0]
    knn_imputer = KNNImputer(n_neighbors=n_value)
    colsNumeric= df[cols_with_null].select_dtypes(include=[np.number])
    df[colsNumeric.columns] = knn_imputer.fit_transform(colsNumeric)
    return df
    
def iterative_imputer(df):
    cols_with_null= df.columns[df.isnull().sum()>0]
    numeric_iterative= df[cols_with_null].select_dtypes(include=[np.number]).columns
    imputer = IterativeImputer(max_iter=10, random_state=0)
    df[numeric_iterative] = imputer.fit_transform(df[numeric_iterative])
    return df

def outliers_z_score(df):
    num_cols = df.select_dtypes(include=[np.number])
    z_scores = np.abs(stats.zscore(num_cols))
    z_scores = pd.DataFrame(z_scores, columns=num_cols.columns, index=num_cols.index)
    outliers = z_scores[(z_scores > 3).any(axis=1)]
    df.drop(outliers.index,inplace=True)
    return df

def outliers_IQR(df):
    numD=df.select_dtypes(include='number')
    Q1 = numD.quantile(0.25)
    Q3 = numD.quantile(0.75)
    IQR = Q3 - Q1
    outliers_IQR = (numD < (Q1 - 1.5 * IQR)) | (numD > (Q3 + 1.5 * IQR))
    df.drop(outliers_IQR[outliers_IQR.any(axis=1)].index, inplace=True)
    return df

def label_encoder(df,col_names):
    encoder=LabelEncoder()
    for i in col_names:
        df[i]=encoder.fit_transform(df[i])
    return df
    
def one_hot_encoder(df, col_names):
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    encoded_data = encoder.fit_transform(df[col_names])
    encoded_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out(df[col_names].columns))
    df = df.drop(col_names, axis=1)
    df = pd.concat([df.reset_index(drop=True), encoded_df], axis=1)
    return df
    
def binary_encoding(df,col_names):
    encoder = ce.BinaryEncoder(cols=col_names)
    encoded_state = encoder.fit_transform(df[col_names])
    df.drop(col_names, axis=1,inplace=True)
    df = pd.concat([df, encoded_state], axis=1)
    return df

def ordinal_encoder(df, col_names):
    for i in col_names:
        encoder = ce.OrdinalEncoder(cols=[i])
        df[i] = encoder.fit_transform(df[[i]])
    return df

def minMax_scaler(df):
    scaler = MinMaxScaler()
    data_scaled = scaler.fit_transform(df) 
    df = pd.DataFrame(data_scaled, columns=df.columns,index=df.index) 
    return df
    
def standard_scaler(df):
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df) 
    df = pd.DataFrame(scaled_data, columns=df.columns)
    return df
    
    